# CCA-F Mock Exam — Question Generator

Generates a fresh set of 25 exam-grade questions for the **Claude Certified Architect – Foundations (CCA-F)** exam,  
then exports them to a structured `.md` file with solutions and anti-pattern analysis.

---
## Setup
```bash
pip install anthropic
```

In [1]:
# ── 1. CONFIGURATION ─────────────────────────────────────────────────────────
import os

# Set your Anthropic API key here or export it as an environment variable:
# export ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")

# ── Model selection ───────────────────────────────────────────────────────────
# Haiku  → fast, cheap,   good for iteration
# Sonnet → balanced,      recommended for quality
# Opus   → highest quality, slowest, most expensive
MODEL = "claude-haiku-4-5-20251001"
# MODEL = "claude-sonnet-4-20250514"
# MODEL = "claude-opus-4-20250514"

# ── Token budget per batch ────────────────────────────────────────────────────
# 5 questions × ~400-500 tokens each = ~2500 comfortably fits Haiku
# Increase to 3500-4000 if using Sonnet/Opus for richer explanations
MAX_TOKENS = 2500

# ── Output file ───────────────────────────────────────────────────────────────
from datetime import datetime
OUTPUT_FILE = f"qna_{datetime.now().strftime('%Y-%m-%dT%H-%M-%S')}.md"

print(f"Model     : {MODEL}")
print(f"Max tokens: {MAX_TOKENS}")
print(f"Output    : {OUTPUT_FILE}")

Model     : claude-haiku-4-5-20251001
Max tokens: 2500
Output    : qna_2026-06-02T12-41-43.md


In [2]:
# ── 2. DOMAIN DEFINITIONS ────────────────────────────────────────────────────

DOMAIN_NAMES = {
    1: "Agentic Architecture & Orchestration",
    2: "Tool Design & MCP Integration",
    3: "Claude Code Configuration & Workflows",
    4: "Prompt Engineering & Structured Output",
    5: "Context Management & Reliability",
}

DOMAIN_WEIGHTS = {1: "27%", 2: "18%", 3: "20%", 4: "20%", 5: "15%"}

# Each batch is a list of (question_id, domain) tuples
# 5 batches × 5 questions = 25 questions total
# Domain distribution: 1→7q, 2→5q, 3→5q, 4→5q, 5→3q
BATCHES = [
    [(1,1),(2,2),(3,1),(4,3),(5,2)],
    [(6,1),(7,4),(8,3),(9,2),(10,4)],
    [(11,1),(12,3),(13,5),(14,4),(15,1)],
    [(16,2),(17,4),(18,5),(19,3),(20,1)],
    [(21,4),(22,5),(23,2),(24,3),(25,1)],
]

print("Domain distribution:")
for d, name in DOMAIN_NAMES.items():
    count = sum(1 for batch in BATCHES for _, dom in batch if dom == d)
    print(f"  Domain {d} ({DOMAIN_WEIGHTS[d]}) — {name}: {count}q")

Domain distribution:
  Domain 1 (27%) — Agentic Architecture & Orchestration: 7q
  Domain 2 (18%) — Tool Design & MCP Integration: 5q
  Domain 3 (20%) — Claude Code Configuration & Workflows: 5q
  Domain 4 (20%) — Prompt Engineering & Structured Output: 5q
  Domain 5 (15%) — Context Management & Reliability: 3q


In [4]:
# ── 3. PROMPT BUILDER ────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "Return ONLY a raw JSON array. "
    "No markdown, no code fences, no explanation. "
    "Start with [ and end with ]."
)


def make_prompt(items: list[tuple[int, int]]) -> str:
    """
    Build the user prompt for a batch of questions.

    Args:
        items: list of (question_id, domain) tuples

    Returns:
        Formatted prompt string
    """
    lines = "\n".join(
        f"ID {qid}: Domain {dom} ({DOMAIN_NAMES[dom]})"
        for qid, dom in items
    )
    ids = [qid for qid, _ in items]

    return f"""Write exactly {len(items)} CCA-F certification exam questions about building Claude-based AI systems.

Questions:
{lines}

Requirements:
- Generic SaaS/e-commerce/devtools/enterprise scenarios only.
- 4 options each, exactly one correct answer.
- Wrong options must be plausible mistakes, not obviously wrong.
- Test architectural judgment, not API memorisation.
- Scenarios to use: Customer Support Agent, E-commerce Platform, Multi-Agent Pipeline,
  Developer Productivity Tool, Code Review CI/CD, Content Moderation, Data Extraction,
  Enterprise Knowledge Base, Marketing Automation.
- Every question must be distinct — no repeated concepts across the batch.
- Difficulty: medium to hard, scenario-based.
- "explanation": 1-2 sentences why the correct answer is right.
- "antipattern_index": index (0-3) of the most dangerous wrong option (must differ from "answer").
- "antipattern_reason": 1 sentence why it is an anti-pattern. Use Claude terminology where
  applicable: context bloat, non-idempotent tool, missing circuit breaker, unbounded retry,
  prompt injection surface, context saturation, raw object parameter.

Output a JSON array of exactly {len(items)} objects with IDs {ids}:
[{{"id":N,"domain":N,"scenario":"name","question":"text","options":["A","B","C","D"],
  "answer":N,"explanation":"text","antipattern_index":N,"antipattern_reason":"text"}}]"""


# Preview the prompt for batch 1
print(make_prompt(BATCHES[0]))

Write exactly 5 CCA-F certification exam questions about building Claude-based AI systems.

Questions:
ID 1: Domain 1 (Agentic Architecture & Orchestration)
ID 2: Domain 2 (Tool Design & MCP Integration)
ID 3: Domain 1 (Agentic Architecture & Orchestration)
ID 4: Domain 3 (Claude Code Configuration & Workflows)
ID 5: Domain 2 (Tool Design & MCP Integration)

Requirements:
- Generic SaaS/e-commerce/devtools/enterprise scenarios only.
- 4 options each, exactly one correct answer.
- Wrong options must be plausible mistakes, not obviously wrong.
- Test architectural judgment, not API memorisation.
- Scenarios to use: Customer Support Agent, E-commerce Platform, Multi-Agent Pipeline,
  Developer Productivity Tool, Code Review CI/CD, Content Moderation, Data Extraction,
  Enterprise Knowledge Base, Marketing Automation.
- Every question must be distinct — no repeated concepts across the batch.
- Difficulty: medium to hard, scenario-based.
- "explanation": 1-2 sentences why the correct answer

In [5]:
# ── 4. QUESTION GENERATION ───────────────────────────────────────────────────
import anthropic
import json
import time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_batch(items: list[tuple[int, int]], retries: int = 2) -> list[dict]:
    """
    Send one batch to Claude and return parsed questions.
    Retries on JSON parse failure (truncation is rare at 5q/2500 tokens).
    """
    for attempt in range(retries + 1):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": make_prompt(items)}],
            )
            raw = response.content[0].text.strip()
            # Strip any accidental markdown fences
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            if attempt < retries:
                print(f"    JSON parse failed (attempt {attempt+1}), retrying... ({e})")
                time.sleep(2)
            else:
                raise RuntimeError(f"Batch failed after {retries+1} attempts: {e}")

In [6]:
def generate_all_questions() -> list[dict]:
    """Run all 5 batches sequentially, saving progress after each."""
    all_questions = []

    for i, batch in enumerate(BATCHES):
        print(f"Batch {i+1}/{len(BATCHES)} — generating {len(batch)} questions "
              f"(IDs {batch[0][0]}–{batch[-1][0]})...", end=" ", flush=True)
        try:
            questions = call_batch(batch)
            all_questions.extend(questions)
            print(f"✓  ({len(all_questions)} total saved)")
        except Exception as e:
            print(f"\n✗ FAILED: {e}")
            print(f"  {len(all_questions)} questions saved from {i} completed batch(es).")
            print("  Re-run this cell to retry from the beginning, or handle partial below.")
            break

    # Re-index IDs 1–N in order
    for idx, q in enumerate(all_questions):
        q["id"] = idx + 1

    return all_questions

In [7]:
questions = generate_all_questions()
print(f"\n{'='*50}")
print(f"Generation complete: {len(questions)} questions")

Batch 1/5 — generating 5 questions (IDs 1–5)... ✓  (5 total saved)
Batch 2/5 — generating 5 questions (IDs 6–10)... ✓  (10 total saved)
Batch 3/5 — generating 5 questions (IDs 11–15)... ✓  (15 total saved)
Batch 4/5 — generating 5 questions (IDs 16–20)... ✓  (20 total saved)
Batch 5/5 — generating 5 questions (IDs 21–25)... ✓  (25 total saved)

Generation complete: 25 questions


In [8]:
# ── 5. PREVIEW SAMPLE QUESTIONS ──────────────────────────────────────────────

LABELS = ["A", "B", "C", "D"]

def preview_question(q: dict) -> None:
    print(f"Q{q['id']}. [{DOMAIN_NAMES[q['domain']]}] — {q['scenario']}")
    print(f"   {q['question']}")
    for i, opt in enumerate(q["options"]):
        marker = "✓" if i == q["answer"] else ("⚠" if i == q.get("antipattern_index") else " ")
        print(f"   {marker} {LABELS[i]}. {opt}")
    print(f"   → {q['explanation']}")
    print(f"   ⚠ Anti-pattern ({LABELS[q.get('antipattern_index',0)]}): {q.get('antipattern_reason','')}")
    print()

# Show first 3 questions as a sanity check
for q in questions[:3]:
    preview_question(q)

Q1. [Agentic Architecture & Orchestration] — Multi-Agent Pipeline
   You are designing a multi-agent system for an e-commerce platform that processes customer orders through three sequential agents: validation, enrichment, and fulfillment. The validation agent sometimes rejects orders due to temporary payment service timeouts. How should you structure agent handoff to prevent cascading failures?
   ✓ A. Implement exponential backoff with jitter in the validation agent and make handoff conditional on success status codes only
   ⚠ B. Have the orchestrator retry the entire pipeline from the start whenever any agent fails
     C. Configure each agent to independently handle retries up to 10 times before passing to the next agent
     D. Store intermediate validation results in a shared cache so failed orders can skip validation on retry
   → Conditional handoff with proper backoff respects agent boundaries and prevents state corruption, while retrying from the start or allowing unbounded 

In [9]:
# ── 6. DOMAIN DISTRIBUTION CHECK ─────────────────────────────────────────────
from collections import Counter

dist = Counter(q["domain"] for q in questions)
print("Domain distribution in generated set:")
for d in sorted(dist):
    bar = "█" * dist[d]
    print(f"  Domain {d} ({DOMAIN_WEIGHTS[d]}) {bar} {dist[d]}q — {DOMAIN_NAMES[d]}")

Domain distribution in generated set:
  Domain 1 (27%) ███████ 7q — Agentic Architecture & Orchestration
  Domain 2 (18%) █████ 5q — Tool Design & MCP Integration
  Domain 3 (20%) █████ 5q — Claude Code Configuration & Workflows
  Domain 4 (20%) █████ 5q — Prompt Engineering & Structured Output
  Domain 5 (15%) ███ 3q — Context Management & Reliability


In [10]:
# ── 7. MARKDOWN EXPORT ───────────────────────────────────────────────────────

def build_markdown(questions: list[dict]) -> str:
    ts = datetime.now().strftime("%d %b %Y, %I:%M %p")

    # Part 1 — Questions
    q_sections = []
    for i, q in enumerate(questions):
        opts = "\n".join(f"  - **{LABELS[oi]}.** {opt}" for oi, opt in enumerate(q["options"]))
        q_sections.append(
            f"### Q{i+1}. [Domain {q['domain']} — {DOMAIN_NAMES[q['domain']]}]\n"
            f"**Scenario:** {q['scenario']}\n\n"
            f"{q['question']}\n\n{opts}"
        )

    # Part 2 — Solutions
    s_sections = []
    for i, q in enumerate(questions):
        preview = q["question"][:75] + ("…" if len(q["question"]) > 75 else "")
        ap_label  = LABELS[q.get("antipattern_index", 0)]
        ap_option = q["options"][q.get("antipattern_index", 0)]
        ap_reason = q.get("antipattern_reason", "_Not available_")
        s_sections.append(
            f"### Q{i+1}. {preview}\n\n"
            f"**✅ Correct Answer: {LABELS[q['answer']]}**\n"
            f"> {q['options'][q['answer']]}\n\n"
            f"{q['explanation']}\n\n"
            f"**⚠️ Anti-Pattern: {ap_label}**\n"
            f"> {ap_option}\n\n"
            f"{ap_reason}"
        )

    return (
        f"# CCA-F Mock Exam — Questions & Solutions\n"
        f"_Generated: {ts} · Model: {MODEL}_\n\n"
        f"---\n\n"
        f"## Part 1 — Questions\n\n"
        + "\n\n---\n\n".join(q_sections)
        + "\n\n---\n\n"
        f"## Part 2 — Solutions\n\n"
        + "\n\n---\n\n".join(s_sections)
        + "\n"
    )


md_content = build_markdown(questions)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(md_content)

print(f"✓ Saved {len(questions)} questions to: {OUTPUT_FILE}")
print(f"  File size: {len(md_content):,} characters")

✓ Saved 25 questions to: qna_2026-06-02T12-41-43.md
  File size: 45,452 characters


In [11]:
# ── 8. DOWNLOAD IN JUPYTER / COLAB ───────────────────────────────────────────
# Run whichever block matches your environment.

import os

if os.path.exists("/content"):  # Google Colab
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"Downloading {OUTPUT_FILE} via Colab...")
else:  # Local Jupyter
    abs_path = os.path.abspath(OUTPUT_FILE)
    print(f"File saved locally at:\n  {abs_path}")
    print("Open it in any Markdown viewer, Obsidian, or VS Code.")

File saved locally at:
  /Users/rajeevkulkarni/Downloads/qna_2026-06-02T12-41-43.md
Open it in any Markdown viewer, Obsidian, or VS Code.


In [12]:
# ── 9. (OPTIONAL) SAVE RAW JSON ──────────────────────────────────────────────
# Useful if you want to load questions back into the web app or do further analysis.

json_file = OUTPUT_FILE.replace(".md", ".json")
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2)

print(f"✓ Raw JSON saved to: {json_file}")

✓ Raw JSON saved to: qna_2026-06-02T12-41-43.json
